<a href="https://colab.research.google.com/github/Kingelanci/graphysics/blob/main/pali_stylometry_complete_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PÄli Canon Stylometric Analysis v3 - COMPLETE

**Versione completa con:**
- Training corpora definiti
- Ablation tests (verse vs verse, verse vs prose, prose vs prose)
- Canon-wide analysis
- Terminology verification
- Report finale DETTAGLIATISSIMO

---

**TRAINING CORPORA:**
- **EARLY CORE**: Aá¹­á¹­hakavagga (Snp 4) + PÄrÄyanavagga (Snp 5) + Snp 1.3, 1.12, 3.6, 3.11
- **LATE CORE**: Buddhavaá¹ƒsa (bv) + Petavatthu (pv) + VimÄnavatthu (vv)

**EXCLUSIONS**: Niddesa texts (mnd, cnd) - canonical commentaries on Snp 4-5

In [ ]:
!pip install -q scikit-learn pandas numpy

In [ ]:
import json
import os
import gc
import re
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print("âœ“ Imports OK")

âœ“ Imports OK


## 1. Download Canon

In [ ]:
!rm -rf bilara-data
!git clone --depth 1 --filter=blob:none --sparse https://github.com/suttacentral/bilara-data.git
%cd bilara-data
!git sparse-checkout set root/pli/ms/sutta
%cd ..
print("âœ“ Downloaded")

Cloning into 'bilara-data'...
remote: Enumerating objects: 1736, done.
remote: Counting objects: 100% (1736/1736), done.
remote: Compressing objects: 100% (1552/1552), done.
remote: Total 1736 (delta 10), reused 979 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (1736/1736), 1.10 MiB | 12.56 MiB/s, done.
Resolving deltas: 100% (10/10), done.
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 10 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), 60.23 KiB | 5.47 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/bilara-data
remote: Enumerating objects: 5764, done.
remote: Counting objects: 100% (5764/5764), done.
remote: Compressing objects: 100% (5484/5484), done.
remote: Total 5764 (delta 282), reused 5687 (delta 280), pack-reused 0 (from 0)
Receiving objects: 100% (5764/5764), 5.61 MiB | 4.24 MiB/s, done.
Resolving deltas: 100% (282/282

In [ ]:
BILARA_ROOT = Path("bilara-data/root/pli/ms/sutta")
EXCLUDE_COLLECTIONS = {'mnd', 'cnd'}  # Niddesa

for d in sorted(BILARA_ROOT.iterdir()):
    if d.is_dir():
        n = len(list(d.glob("**/*_root-pli-ms.json")))
        print(f"{d.name}: {n} files")

an: 1408 files
dn: 34 files
kn: 2351 files
mn: 152 files
sn: 1819 files


## 2. Data Loading Functions

In [ ]:
def load_sutta_pattern(pattern: str) -> list:
    """Load segments matching a pattern."""
    segments = []

    if re.match(r'^snp[0-5]$', pattern):
        base = BILARA_ROOT / "kn" / "snp"
        for f in base.glob(f"**/{pattern}.*_root-pli-ms.json"):
            with open(f) as fp:
                data = json.load(fp)
            for seg_id, text in data.items():
                if text and len(str(text).split()) >= 2:
                    segments.append({"segment_id": seg_id, "text": text, "source": pattern})
    else:
        if pattern.startswith("snp"):
            base = BILARA_ROOT / "kn" / "snp"
        elif pattern.startswith("dn"):
            base = BILARA_ROOT / "dn"
        else:
            base = BILARA_ROOT / "kn"

        for f in base.glob(f"**/{pattern}_root-pli-ms.json"):
            with open(f) as fp:
                data = json.load(fp)
            for seg_id, text in data.items():
                if text and len(str(text).split()) >= 2:
                    segments.append({"segment_id": seg_id, "text": text, "source": pattern})

    return segments


def load_collection(coll: str, base_path=None) -> list:
    """Load entire collection."""
    if base_path is None:
        base = BILARA_ROOT / "kn" / coll
    else:
        base = base_path
    if not base.exists():
        return []

    segments = []
    for f in sorted(base.glob("**/*_root-pli-ms.json")):
        # Skip Niddesa
        rel_path = str(f.relative_to(BILARA_ROOT))
        if any(f"/{exc}/" in rel_path for exc in EXCLUDE_COLLECTIONS):
            continue

        with open(f) as fp:
            data = json.load(fp)
        for seg_id, text in data.items():
            if text and len(str(text).split()) >= 2:
                segments.append({"segment_id": seg_id, "text": text, "source": coll, "file": str(f)})

    return segments


def load_nikaya(nikaya: str) -> list:
    """Load entire nikaya."""
    base = BILARA_ROOT / nikaya
    segments = []

    for f in sorted(base.glob("**/*_root-pli-ms.json")):
        rel_path = str(f.relative_to(BILARA_ROOT))
        if any(f"/{exc}/" in rel_path for exc in EXCLUDE_COLLECTIONS):
            continue

        sutta_id = f.stem.replace("_root-pli-ms", "")

        with open(f) as fp:
            data = json.load(fp)
        for seg_id, text in data.items():
            if text and len(str(text).split()) >= 2:
                segments.append({
                    "segment_id": seg_id,
                    "sutta_id": sutta_id,
                    "text": text,
                    "file_path": rel_path,
                    "nikaya": nikaya
                })

    return segments


print("âœ“ Functions ready")

âœ“ Functions ready


## 3. Load Training Data

In [ ]:
# === EARLY CORE ===
early_patterns = ["snp4", "snp5", "snp1.12", "snp1.3", "snp3.11", "snp3.6"]
early_segs = []
for p in early_patterns:
    segs = load_sutta_pattern(p)
    early_segs.extend(segs)
    print(f"  {p}: {len(segs)}")

print(f"\nâ†’ Total EARLY: {len(early_segs)} segments")

  snp4: 893
  snp5: 877
  snp1.12: 72
  snp1.3: 166
  snp3.11: 178
  snp3.6: 220

â†’ Total EARLY: 2406 segments


In [ ]:
# === LATE CORE ===
late_segs = []
for coll in ["bv", "pv", "vv"]:
    segs = load_collection(coll)
    late_segs.extend(segs)
    print(f"  {coll}: {len(segs)}")

print(f"\nâ†’ Total LATE: {len(late_segs)} segments")
print(f"Ratio early:late = 1:{len(late_segs)/len(early_segs):.1f}")

  bv: 3882
  pv: 3366
  vv: 3964

â†’ Total LATE: 11212 segments
Ratio early:late = 1:4.7


## 4. Feature Extraction Functions

In [ ]:
def get_lex_features(texts):
    """Extract lexical features: TTR and avg word length."""
    out = []
    for t in texts:
        words = str(t).split()
        n = len(words) if words else 1
        ttr = len(set(words)) / n
        avg_len = np.mean([len(w) for w in words]) if words else 0
        out.append([ttr, avg_len])
    return np.array(out, dtype=np.float32)


def train_model(train_texts, train_labels, return_components=False):
    """Train logistic regression model with TF-IDF char n-grams."""
    vectorizer = TfidfVectorizer(
        analyzer='char',
        ngram_range=(3, 5),
        max_features=2000,
        min_df=2,
        dtype=np.float32
    )

    X = vectorizer.fit_transform(train_texts).toarray()
    X_lex = get_lex_features(train_texts)
    X = np.hstack([X, X_lex])

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Cross-validation
    model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_scaled, train_labels, cv=cv, scoring='roc_auc')

    # Fit final model
    model.fit(X_scaled, train_labels)

    if return_components:
        return model, vectorizer, scaler, scores
    return scores.mean(), scores.std()


print("âœ“ Feature extraction ready")

âœ“ Feature extraction ready


## 5. MAIN MODEL - Train and Validate

In [ ]:
train_texts = [s['text'] for s in early_segs] + [s['text'] for s in late_segs]
train_labels = [1]*len(early_segs) + [0]*len(late_segs)

print(f"Training set: {sum(train_labels)} early, {len(train_labels)-sum(train_labels)} late")

model, vectorizer, scaler, cv_scores = train_model(train_texts, train_labels, return_components=True)

print(f"\n" + "="*50)
print(f"MAIN MODEL CV AUC: {cv_scores.mean():.3f} Â± {cv_scores.std():.3f}")
print(f"="*50)

# Store for later
MAIN_AUC_MEAN = cv_scores.mean()
MAIN_AUC_STD = cv_scores.std()
N_EARLY = len(early_segs)
N_LATE = len(late_segs)

Training set: 2406 early, 11212 late

MAIN MODEL CV AUC: 0.849 Â± 0.006


## 6. ABLATION TESTS

Three tests to verify independence from genre:
1. **Verse EARLY vs Verse LATE** - same genre, tests chronological signal
2. **Verse EARLY vs Prose LATE** - cross-genre, tests transfer
3. **Prose vs Prose (SN vs AN)** - internal prose comparison

In [ ]:
# Test 1: VERSE EARLY vs VERSE LATE
# Early: Snp 4-5 (verse)
# Late: bv, pv, vv (verse)

print("="*60)
print("ABLATION TEST 1: Verse EARLY vs Verse LATE")
print("="*60)

# Both are already verse
verse_early = early_segs  # Snp 4-5 etc.
verse_late = late_segs    # bv, pv, vv

texts = [s['text'] for s in verse_early] + [s['text'] for s in verse_late]
labels = [1]*len(verse_early) + [0]*len(verse_late)

auc_mean, auc_std = train_model(texts, labels)
print(f"  Verse EARLY: {len(verse_early)} segments")
print(f"  Verse LATE: {len(verse_late)} segments")
print(f"  â†’ AUC: {auc_mean:.2f} (Â±{auc_std:.2f})")

ABLATION_VERSE_VERSE = auc_mean

ABLATION TEST 1: Verse EARLY vs Verse LATE
  Verse EARLY: 2406 segments
  Verse LATE: 11212 segments
  â†’ AUC: 0.85 (Â±0.01)


In [ ]:
# Test 2: VERSE EARLY vs PROSE LATE
# Early: Snp 4-5 (verse)
# Late: DN 14-17 (prose, late redaction)

print("="*60)
print("ABLATION TEST 2: Verse EARLY vs Prose LATE")
print("="*60)

# Load DN 14-17 as prose late
prose_late = []
for sutta in ["dn14", "dn15", "dn16", "dn17"]:
    segs = load_sutta_pattern(sutta)
    prose_late.extend(segs)
    print(f"  Loaded {sutta}: {len(segs)}")

texts = [s['text'] for s in verse_early] + [s['text'] for s in prose_late]
labels = [1]*len(verse_early) + [0]*len(prose_late)

auc_mean, auc_std = train_model(texts, labels)
print(f"\n  Verse EARLY: {len(verse_early)} segments")
print(f"  Prose LATE (DN14-17): {len(prose_late)} segments")
print(f"  â†’ AUC: {auc_mean:.2f} (Â±{auc_std:.2f})")

ABLATION_VERSE_PROSE = auc_mean

ABLATION TEST 2: Verse EARLY vs Prose LATE
  Loaded dn14: 844
  Loaded dn15: 276
  Loaded dn16: 1642
  Loaded dn17: 441

  Verse EARLY: 2406 segments
  Prose LATE (DN14-17): 3203 segments
  â†’ AUC: 0.97 (Â±0.00)


In [ ]:
# Test 3: PROSE vs PROSE (SN vs AN)
# Using SN as more archaic, AN as more scholastic

print("="*60)
print("ABLATION TEST 3: Prose vs Prose (SN vs AN)")
print("="*60)

# Sample from SN and AN
sn_sample = load_nikaya('sn')[:10000]  # Sample
an_sample = load_nikaya('an')[:10000]  # Sample

texts = [s['text'] for s in sn_sample] + [s['text'] for s in an_sample]
labels = [1]*len(sn_sample) + [0]*len(an_sample)

auc_mean, auc_std = train_model(texts, labels)
print(f"  SN sample: {len(sn_sample)} segments")
print(f"  AN sample: {len(an_sample)} segments")
print(f"  â†’ AUC: {auc_mean:.2f} (Â±{auc_std:.2f})")

ABLATION_PROSE_PROSE = auc_mean

del sn_sample, an_sample
gc.collect()

ABLATION TEST 3: Prose vs Prose (SN vs AN)
  SN sample: 10000 segments
  AN sample: 10000 segments
  â†’ AUC: 0.93 (Â±0.00)


185

## 7. Score Entire Canon

In [ ]:
def score_segments(segments, vectorizer, scaler, model):
    """Score a list of segments."""
    if not segments:
        return []

    texts = [s['text'] for s in segments]
    X = vectorizer.transform(texts).toarray()
    X_lex = get_lex_features(texts)
    X = np.hstack([X, X_lex])
    X = scaler.transform(X)
    probs = model.predict_proba(X)[:, 1]

    for i, s in enumerate(segments):
        s['p_early'] = float(probs[i])

    return segments


print("Scoring entire canon...")
all_results = []

for nikaya in ['dn', 'mn', 'sn', 'an', 'kn']:
    print(f"\n  Processing {nikaya.upper()}...")
    segments = load_nikaya(nikaya)
    segments = score_segments(segments, vectorizer, scaler, model)

    df = pd.DataFrame(segments)
    df.to_csv(f'{nikaya}_scores.csv', index=False)
    print(f"    âœ“ {len(df):,} segments â†’ {nikaya}_scores.csv")

    all_results.append(df)

print("\nâœ“ Canon scoring complete")

Scoring entire canon...

  Processing DN...
    âœ“ 16,212 segments â†’ dn_scores.csv

  Processing MN...
    âœ“ 26,775 segments â†’ mn_scores.csv

  Processing SN...
    âœ“ 38,997 segments â†’ sn_scores.csv

  Processing AN...
    âœ“ 38,047 segments â†’ an_scores.csv

  Processing KN...
    âœ“ 120,322 segments â†’ kn_scores.csv

âœ“ Canon scoring complete


In [ ]:
# Create sutta summary
all_data = pd.concat(all_results, ignore_index=True)

sutta_summary = all_data.groupby(['sutta_id', 'nikaya']).agg({
    'p_early': ['mean', 'std', 'min', 'max', 'count']
}).reset_index()
sutta_summary.columns = ['sutta_id', 'nikaya', 'mean', 'std', 'min', 'max', 'n_segs']
sutta_summary['range'] = sutta_summary['max'] - sutta_summary['min']
sutta_summary.to_csv('all_suttas_summary.csv', index=False)

print(f"Total segments: {len(all_data):,}")
print(f"Total suttas: {len(sutta_summary):,}")

Total segments: 240,353
Total suttas: 5,725


## 8. NIKÄ€YA ANALYSIS

In [ ]:
print("="*70)
print("NIKÄ€YA SUMMARY STATISTICS")
print("="*70)

nikaya_stats = []
for nik in ['dn', 'mn', 'sn', 'an', 'kn']:
    nik_data = all_data[all_data['nikaya'] == nik]
    nik_summary = sutta_summary[sutta_summary['nikaya'] == nik]
    nikaya_stats.append({
        'nikaya': nik.upper(),
        'mean_p': nik_summary['mean'].mean(),
        'median_p': nik_summary['mean'].median(),
        'std_p': nik_summary['mean'].std(),
        'n_suttas': len(nik_summary),
        'n_segments': len(nik_data)
    })

nikaya_df = pd.DataFrame(nikaya_stats).sort_values('mean_p', ascending=False)
print(nikaya_df.to_string(index=False))

NIKAYA_STATS = nikaya_df.copy()

NIKÄ€YA SUMMARY STATISTICS
nikaya   mean_p  median_p    std_p  n_suttas  n_segments
    MN 0.509477  0.507279 0.119646       152       26775
    DN 0.488534  0.487801 0.135568        34       16212
    SN 0.340209  0.336036 0.179380      1819       38997
    AN 0.325702  0.310849 0.186882      1408       38047
    KN 0.252551  0.220794 0.182013      2312      120322


In [ ]:
# Khuddaka breakdown
print("\n" + "="*70)
print("KHUDDAKA NIKÄ€YA BREAKDOWN BY COLLECTION")
print("="*70)

kn_data = all_data[all_data['nikaya'] == 'kn'].copy()
kn_data['collection'] = kn_data['file_path'].str.split('/').str[1]

kn_breakdown = kn_data.groupby('collection')['p_early'].agg(['mean', 'median', 'std', 'count'])
kn_breakdown = kn_breakdown.sort_values('mean', ascending=False)
print(kn_breakdown.to_string())

KN_BREAKDOWN = kn_breakdown.copy()


KHUDDAKA NIKÄ€YA BREAKDOWN BY COLLECTION
                mean    median       std  count
collection                                     
snp         0.564624  0.688935  0.405931   5527
pe          0.548408  0.660244  0.423847   4583
ps          0.476239  0.442018  0.420018  13837
ne          0.456104  0.329650  0.426291   4170
ud          0.438608  0.314006  0.412182   2490
mil         0.396143  0.153037  0.421393   7243
dhp         0.333870  0.079255  0.397071   1760
iti         0.305272  0.095874  0.373975   2413
thag        0.290347  0.049588  0.370425   5659
ja          0.259580  0.031572  0.362203  28299
kp          0.256627  0.065841  0.332781    342
thig        0.221090  0.018463  0.338716   2203
cp          0.147025  0.006825  0.274361   1515
thi-ap      0.142912  0.003507  0.280932   4826
tha-ap      0.124203  0.003297  0.259839  24243
pv          0.064406  0.004010  0.140881   3366
vv          0.043782  0.000719  0.119206   3964
bv          0.042042  0.002447  0.110607   388

## 9. TERMINOLOGY ANALYSIS

In [ ]:
def analyze_term(term, data, regex=False):
    """Analyze occurrences of a term."""
    if regex:
        matches = data[data['text'].str.contains(term, case=False, na=False, regex=True)]
    else:
        matches = data[data['text'].str.contains(term, case=False, na=False)]

    if len(matches) == 0:
        return None

    return {
        'term': term,
        'n': len(matches),
        'mean_p': matches['p_early'].mean(),
        'std_p': matches['p_early'].std(),
        'min_p': matches['p_early'].min(),
        'max_p': matches['p_early'].max()
    }


print("="*70)
print("TABLE 6: CONSCIOUSNESS TERMINOLOGY BY STRATUM")
print("="*70)

consciousness_terms = [
    ('tadupÄdÄnaá¹', False),
    ('tannissitaá¹', False),
    ('appatiá¹­á¹­hita', False),
    (r'\banattÄ\b', True),
    ('rÅ«paá¹ anattÄ', False),
    ('pabhassara', False),
    ('bhavaá¹…ga', False),
]

table6_results = []
for term, regex in consciousness_terms:
    result = analyze_term(term, all_data, regex)
    if result:
        table6_results.append(result)
        print(f"{term:<30} P={result['mean_p']:.2f} N={result['n']}")

TABLE6 = pd.DataFrame(table6_results)

TABLE 6: CONSCIOUSNESS TERMINOLOGY BY STRATUM
pabhassara                     P=0.53 N=48


In [ ]:
print("\n" + "="*70)
print("TABLE 7: DIá¹¬á¹¬HI (VIEWS) TERMINOLOGY")
print("="*70)

ditthi_terms = [
    ('diá¹­á¹­hiá¹ pahÄya', False),
    ('micchÄdiá¹­á¹­hi', False),
    ('sammÄdiá¹­á¹­hi', False),
]

table7_results = []
for term, regex in ditthi_terms:
    result = analyze_term(term, all_data, regex)
    if result:
        table7_results.append(result)
        print(f"{term:<30} P={result['mean_p']:.2f} N={result['n']}")

TABLE7 = pd.DataFrame(table7_results)


TABLE 7: DIá¹¬á¹¬HI (VIEWS) TERMINOLOGY


In [ ]:
print("\n" + "="*70)
print("TABLE 4: NON-CIRCULARITY TEST (Terms absent from training)")
print("="*70)

# These terms should NOT appear in Snp 4-5 training data
non_circ_terms = [
    ('sammÄdiá¹­á¹­hi', False),
    ('arahant', False),
    ('bojjhaá¹…ga', False),
    ('rÅ«paá¹ anattÄ', False),
]

table4_results = []
for term, regex in non_circ_terms:
    result = analyze_term(term, all_data, regex)
    if result:
        table4_results.append(result)
        print(f"{term:<30} P={result['mean_p']:.2f} N={result['n']}")

TABLE4 = pd.DataFrame(table4_results)


TABLE 4: NON-CIRCULARITY TEST (Terms absent from training)
arahant                        P=0.44 N=450


In [ ]:
print("\n" + "="*70)
print("TABLE 5: TERMINOLOGICAL SUBSTITUTIONS (Archaic â†’ Scholastic)")
print("="*70)

substitution_pairs = [
    ('tannissitaá¹', 'rÅ«paá¹ anattÄ'),
    ('diá¹­á¹­hiá¹ pahÄya', 'sammÄdiá¹­á¹­hi'),
    (r'\bmuni\b', 'arahant'),
    ('akiÃ±cana', 'bojjhaá¹…ga'),
]

table5_results = []
for archaic, scholastic in substitution_pairs:
    r1 = analyze_term(archaic, all_data, regex='\\b' in archaic)
    r2 = analyze_term(scholastic, all_data)
    if r1 and r2:
        delta = r1['mean_p'] - r2['mean_p']
        table5_results.append({
            'archaic': archaic, 'archaic_p': r1['mean_p'], 'archaic_n': r1['n'],
            'scholastic': scholastic, 'scholastic_p': r2['mean_p'], 'scholastic_n': r2['n'],
            'delta': delta
        })
        print(f"{archaic:<20} P={r1['mean_p']:.2f} (N={r1['n']}) â†’ {scholastic:<20} P={r2['mean_p']:.2f} (N={r2['n']}) Î”={delta:.2f}")

TABLE5 = pd.DataFrame(table5_results)


TABLE 5: TERMINOLOGICAL SUBSTITUTIONS (Archaic â†’ Scholastic)
\bmuni\b             P=0.58 (N=179) â†’ arahant              P=0.44 (N=450) Î”=0.13


## 10. FRAME DETECTION

In [ ]:
print("="*70)
print("FRAME DETECTION ANALYSIS")
print("="*70)

# "Evaá¹ me sutaá¹" formula
evam = all_data[all_data['text'].str.contains('evaá¹ me sutaá¹', case=False, na=False)]
print(f"\n'Evaá¹ me sutaá¹' formula:")
print(f"  N = {len(evam)}")
print(f"  Mean P = {evam['p_early'].mean():.2f}")

# Very short segments (titles)
short = all_data[all_data['text'].str.len() < 20]
print(f"\nTitle/header segments (<20 chars):")
print(f"  N = {len(short):,}")
print(f"  Mean P = {short['p_early'].mean():.2f}")

FRAME_EVAM = {'n': len(evam), 'mean_p': evam['p_early'].mean()}
FRAME_SHORT = {'n': len(short), 'mean_p': short['p_early'].mean()}

FRAME DETECTION ANALYSIS

'Evaá¹ me sutaá¹' formula:
  N = 0
  Mean P = nan

Title/header segments (<20 chars):
  N = 19,293
  Mean P = 0.24


## 11. PARADIGMATIC PASSAGES

In [ ]:
print("="*70)
print("PARADIGMATIC EARLY-STYLE PASSAGES")
print("="*70)

# SN 1.1 - Oghataraá¹‡a
print("\n--- SN 1.1 (Oghataraá¹‡a) ---")
sn1_1 = all_data[all_data['sutta_id'] == 'sn1.1'].sort_values('segment_id')
print(f"Total segments: {len(sn1_1)}, Mean P: {sn1_1['p_early'].mean():.3f}")
print("\nKey passage (appatiá¹­á¹­haá¹ anÄyÅ«haá¹):")
key = sn1_1[sn1_1['text'].str.contains('ppatiá¹­á¹­haá¹', case=False, na=False)]
for _, row in key.iterrows():
    print(f"  P={row['p_early']:.2f}: {row['text'][:80]}")

# SN 22.53-55
print("\n--- SN 22.53-55 (Upaya suttas) ---")
for sid in ['sn22.53', 'sn22.54', 'sn22.55']:
    s = all_data[all_data['sutta_id'] == sid]
    if len(s) > 0:
        print(f"  {sid}: {len(s)} segments, mean P = {s['p_early'].mean():.3f}")

# appatiá¹­á¹­hita viÃ±Ã±Äá¹‡a
print("\nKey term 'appatiá¹­á¹­hita viÃ±Ã±Äá¹‡a':")
key = all_data[all_data['text'].str.contains('appatiá¹­á¹­hita', case=False, na=False) &
               all_data['text'].str.contains('viÃ±Ã±Äá¹‡a', case=False, na=False)]
print(f"  N = {len(key)}, Mean P = {key['p_early'].mean():.2f}")

PARADIGMATIC EARLY-STYLE PASSAGES

--- SN 1.1 (Oghataraá¹‡a) ---
Total segments: 19, Mean P: 0.228

Key passage (appatiá¹­á¹­haá¹ anÄyÅ«haá¹):

--- SN 22.53-55 (Upaya suttas) ---
  sn22.53: 19 segments, mean P = 0.381
  sn22.54: 30 segments, mean P = 0.389
  sn22.55: 84 segments, mean P = 0.651

Key term 'appatiá¹­á¹­hita viÃ±Ã±Äá¹‡a':
  N = 0, Mean P = nan


In [ ]:
# Top early-style segments
print("\n" + "="*70)
print("TOP 20 EARLY-STYLE SEGMENTS (P > 0.99)")
print("="*70)

# Exclude training data
train_pattern = r'^(snp[45]\.|snp1\.12|snp1\.3|snp3\.11|snp3\.6|bv|pv|vv)'
grey_data = all_data[~all_data['sutta_id'].str.match(train_pattern)]

top_early = grey_data.nlargest(20, 'p_early')
for _, row in top_early.iterrows():
    print(f"P={row['p_early']:.3f} [{row['sutta_id']}]: {row['text'][:70]}")


TOP 20 EARLY-STYLE SEGMENTS (P > 0.99)
P=1.000 [ja528]: vasa brāhmaṇa māgamā”. 
P=1.000 [ja343]: vasa kuntini māgamā”. 
P=1.000 [ja492]: Bhayaṭṭitā leṇagavesino puthu; 
P=1.000 [an7.96-614]: ghānaviññāṇe … 
P=1.000 [mil7.7.8]: 8. Māgavikaṅgapañha 
P=1.000 [ps1.5]: Evaṁ siyā eko ñāṇavimokkho dasa ñāṇavimokkhā honti, dasa ñāṇavimokkhā 
P=1.000 [ps1.5]: Evaṁ siyā eko ñāṇavimokkho dasa ñāṇavimokkhā honti, dasa ñāṇavimokkhā 
P=1.000 [ps1.5]: Evaṁ siyā eko ñāṇavimokkho dasa ñāṇavimokkhā honti, dasa ñāṇavimokkhā 
P=1.000 [ja466]: Appampi nācceti sa bhūripañño. 
P=1.000 [ps1.5]: Jhāte ca jhāpe ca jānātīti—jhānavimokkho—ayaṁ jhānavimokkho. 
P=1.000 [mn107]: “yeme, bho gotama, puggalā assaddhā jīvikatthā na saddhā agārasmā anag
P=1.000 [ud6.6]: Mānaganthā mānavinibaddhā; 
P=1.000 [ja283]: Bhayaṭṭitā leṇagavesino puthū; 
P=1.000 [ne37]: Ārammaṇametaṁ na hoti viññāṇassa ṭhitiyā, ārammaṇe asati patiṭṭhā viññ
P=1.000 [pe5]: Anissitassa calitaṁ natthīti tassa evaṁ diṭṭhiyā taṇhāya ca pahānaṁ ta
P=1.

## 12. TOP SUTTAS ANALYSIS

In [ ]:
# Exclude training data
train_pattern = r'^(snp[45]\.|snp1\.12|snp1\.3|snp3\.11|snp3\.6|bv|pv|vv|mnd|cnd)'
grey = sutta_summary[~sutta_summary['sutta_id'].str.match(train_pattern)].copy()
grey = grey[grey['n_segs'] >= 5]

print("="*70)
print(f"GREY ZONE ANALYSIS ({len(grey)} suttas, excluding training/niddesa)")
print("="*70)

print("\nTOP 30 EARLY-STYLE SUTTAS:")
print("-"*50)
top_early_suttas = grey.nlargest(30, 'mean')
for _, r in top_early_suttas.iterrows():
    print(f"{r['sutta_id']:<25} {r['nikaya']:<4} P={r['mean']:.3f} (n={int(r['n_segs'])})")

TOP_EARLY_SUTTAS = top_early_suttas.copy()

GREY ZONE ANALYSIS (5316 suttas, excluding training/niddesa)

TOP 30 EARLY-STYLE SUTTAS:
--------------------------------------------------
ps1.9                     kn   P=0.921 (n=22)
sn35.162                  sn   P=0.921 (n=22)
an10.82                   an   P=0.909 (n=25)
an8.77                    an   P=0.862 (n=49)
sn35.118                  sn   P=0.857 (n=26)
sn35.70                   sn   P=0.856 (n=31)
sn46.8                    sn   P=0.855 (n=10)
an5.193                   an   P=0.852 (n=49)
sn35.131                  sn   P=0.845 (n=26)
sn45.3                    sn   P=0.844 (n=13)
an7.54                    an   P=0.841 (n=37)
sn33.5                    sn   P=0.837 (n=13)
sn33.4                    sn   P=0.837 (n=13)
sn35.165                  sn   P=0.835 (n=11)
mil6.2.8                  kn   P=0.829 (n=6)
sn35.79                   sn   P=0.829 (n=14)
sn35.27                   sn   P=0.826 (n=16)
an6.68                    an   P=0.823 (n=14)
sn35.124                  sn   P=

In [ ]:
print("\nTOP 30 LATE-STYLE SUTTAS:")
print("-"*50)
top_late_suttas = grey.nsmallest(30, 'mean')
for _, r in top_late_suttas.iterrows():
    print(f"{r['sutta_id']:<25} {r['nikaya']:<4} P={r['mean']:.3f} (n={int(r['n_segs'])})")

TOP_LATE_SUTTAS = top_late_suttas.copy()


TOP 30 LATE-STYLE SUTTAS:
--------------------------------------------------
an3.138                   an   P=0.000 (n=6)
an5.13                    an   P=0.000 (n=6)
an7.46                    an   P=0.000 (n=6)
an8.33                    an   P=0.000 (n=6)
an4.265                   an   P=0.000 (n=7)
ja54                      kn   P=0.001 (n=6)
an10.78                   an   P=0.001 (n=15)
an3.149                   an   P=0.001 (n=8)
an5.204                   an   P=0.001 (n=6)
an5.16                    an   P=0.002 (n=9)
an5.59                    an   P=0.002 (n=6)
an10.225-228              an   P=0.002 (n=7)
an8.31                    an   P=0.003 (n=13)
an5.137                   an   P=0.003 (n=10)
an3.46                    an   P=0.004 (n=8)
sn51.9                    sn   P=0.004 (n=14)
an8.27                    an   P=0.004 (n=6)
an8.91-117                an   P=0.004 (n=5)
an5.46                    an   P=0.005 (n=6)
an3.108                   an   P=0.005 (n=8)
an3.146           

In [ ]:
print("\nHIGHEST INTERNAL VARIATION (composite texts?):")
print("-"*50)
high_var = grey[grey['n_segs'] >= 20].nlargest(30, 'range')
for _, r in high_var.iterrows():
    print(f"{r['sutta_id']:<25} {r['nikaya']:<4} range={r['range']:.3f} (min={r['min']:.3f}, max={r['max']:.3f})")


HIGHEST INTERNAL VARIATION (composite texts?):
--------------------------------------------------
ja492                     kn   range=1.000 (min=0.000, max=1.000)
ne37                      kn   range=1.000 (min=0.000, max=1.000)
ps1.1                     kn   range=1.000 (min=0.000, max=1.000)
ja528                     kn   range=1.000 (min=0.000, max=1.000)
pe8                       kn   range=1.000 (min=0.000, max=1.000)
pe1                       kn   range=1.000 (min=0.000, max=1.000)
sn18.12-20                sn   range=1.000 (min=0.000, max=1.000)
ps1.2                     kn   range=1.000 (min=0.000, max=1.000)
pe2                       kn   range=1.000 (min=0.000, max=1.000)
ja530                     kn   range=1.000 (min=0.000, max=1.000)
mn5                       mn   range=1.000 (min=0.000, max=1.000)
dn3                       dn   range=1.000 (min=0.000, max=1.000)
ja505                     kn   range=1.000 (min=0.000, max=1.000)
mil7.3.10                 kn   range=1.000 

## 13. DEEP DIVE FUNCTION

In [ ]:
def deep_dive(sutta_id: str):
    """Detailed analysis of one sutta."""
    sutta_df = all_data[all_data['sutta_id'] == sutta_id].copy()

    if len(sutta_df) == 0:
        print(f"Sutta {sutta_id} not found")
        return

    sutta_df = sutta_df.sort_values('segment_id')

    print(f"\n{'=' * 70}")
    print(f"DEEP DIVE: {sutta_id}")
    print(f"{'=' * 70}")
    print(f"Segments: {len(sutta_df)}")
    print(f"Mean: {sutta_df['p_early'].mean():.3f}")
    print(f"Range: {sutta_df['p_early'].min():.3f} - {sutta_df['p_early'].max():.3f}")

    n = len(sutta_df)
    if n >= 10:
        frame = sutta_df.head(n//10)['p_early'].mean()
        core = sutta_df.iloc[n//5:-n//5]['p_early'].mean() if n > 5 else sutta_df['p_early'].mean()
        print(f"\nFrame (first 10%): {frame:.3f}")
        print(f"Core (middle): {core:.3f}")
        print(f"Difference: {core - frame:+.3f}")

    print(f"\n{'Segment':<25} {'P(early)':<10} {'Text'}")
    print("-" * 80)
    for _, row in sutta_df.iterrows():
        marker = "â–ˆâ–ˆ" if row['p_early'] > 0.7 else "â–“â–“" if row['p_early'] > 0.5 else "â–‘â–‘" if row['p_early'] > 0.3 else "  "
        text = str(row['text'])[:50]
        print(f"{row['segment_id']:<25} {marker} {row['p_early']:.3f}  {text}")


print("âœ“ deep_dive() ready")

âœ“ deep_dive() ready


In [ ]:
deep_dive('sn1.1')


DEEP DIVE: sn1.1
Segments: 19
Mean: 0.228
Range: 0.000 - 0.918

Frame (first 10%): 0.046
Core (middle): 0.291
Difference: +0.245

Segment                   P(early)   Text
--------------------------------------------------------------------------------
sn1.1:0.1                    0.046  Saṁyutta Nikāya 1.1 
sn1.1:0.2                    0.013  1. Naḷavagga 
sn1.1:1.1                 â–ˆâ–ˆ 0.741  Evaṁ me sutaṁ—
sn1.1:1.2                 â–ˆâ–ˆ 0.877  ekaṁ samayaṁ bhagavā sāvatthiyaṁ viharati jetavane
sn1.1:1.3                 â–ˆâ–ˆ 0.785  Atha kho aññatarā devatā abhikkantāya rattiyā abhi
sn1.1:1.4                    0.000  “kathaṁ nu tvaṁ, mārisa, oghamatarī”ti? 
sn1.1:1.5                    0.005  “Appatiṭṭhaṁ khvāhaṁ, āvuso, anāyūhaṁ oghamatarin”
sn1.1:1.6                    0.290  “Yathākathaṁ pana tvaṁ, mārisa, appatiṭṭhaṁ anāyūh
sn1.1:1.7                    0.000  “Yadāsvāhaṁ, āvuso, santiṭṭhāmi tadāssu saṁsīdāmi;
sn1.1:1.8                    0.000  yadāsvāhaṁ, āvuso, āyūhāmi t

In [ ]:
deep_dive('sn22.53')


DEEP DIVE: sn22.53
Segments: 19
Mean: 0.381
Range: 0.000 - 1.000

Frame (first 10%): 0.049
Core (middle): 0.333
Difference: +0.284

Segment                   P(early)   Text
--------------------------------------------------------------------------------
sn22.53:0.1                  0.049  Saṁyutta Nikāya 22.53 
sn22.53:0.2                  0.023  6. Upayavagga 
sn22.53:1.2                  0.177  “Upayo, bhikkhave, avimutto, anupayo vimutto. 
sn22.53:1.3               â–ˆâ–ˆ 0.970  Rūpupayaṁ vā, bhikkhave, viññāṇaṁ tiṭṭhamānaṁ tiṭṭ
sn22.53:1.4                  0.001  Vedanupayaṁ vā …pe… 
sn22.53:1.5                  0.002  saññupayaṁ vā …pe… 
sn22.53:1.6               â–ˆâ–ˆ 0.998  saṅkhārupayaṁ vā, bhikkhave, viññāṇaṁ tiṭṭhamānaṁ 
sn22.53:2.1                  0.002  Yo, bhikkhave, evaṁ vadeyya: 
sn22.53:2.2               â–ˆâ–ˆ 0.999  ‘ahamaññatra rūpā aññatra vedanāya aññatra saññāya
sn22.53:3.1                  0.000  Rūpadhātuyā ce, bhikkhave, bhikkhuno rāgo pahīno h
sn22.53:3.10

---

# 14. FINAL REPORT - COMPREHENSIVE

---

In [ ]:
report = []
report.append("=" * 80)
report.append("PÄ€LI CANON STYLOMETRIC ANALYSIS - FINAL REPORT")
report.append("=" * 80)
report.append("")

# 1. Training Data
report.append("" + "=" * 60)
report.append("1. TRAINING CORPORA")
report.append("=" * 60)
report.append(f"")
report.append(f"EARLY CORE: Aá¹­á¹­hakavagga (Snp 4) + PÄrÄyanavagga (Snp 5) + archaic Snp")
report.append(f"  Sources: snp4, snp5, snp1.12, snp1.3, snp3.11, snp3.6")
report.append(f"  Segments: {N_EARLY:,}")
report.append(f"")
report.append(f"LATE CORE: Buddhavaá¹ƒsa + Petavatthu + VimÄnavatthu")
report.append(f"  Sources: bv, pv, vv")
report.append(f"  Segments: {N_LATE:,}")
report.append(f"")
report.append(f"TOTAL TRAINING: {N_EARLY + N_LATE:,} segments")
report.append(f"Ratio early:late = 1:{N_LATE/N_EARLY:.1f}")
report.append(f"")
report.append(f"EXCLUSIONS: Niddesa texts (mnd, cnd) - canonical commentaries on Snp")

# 2. Model Performance
report.append("")
report.append("=" * 60)
report.append("2. MODEL PERFORMANCE")
report.append("=" * 60)
report.append(f"")
report.append(f"MAIN MODEL (5-fold stratified CV):")
report.append(f"  AUC = {MAIN_AUC_MEAN:.3f} Â± {MAIN_AUC_STD:.3f}")

# 3. Ablation Tests
report.append("")
report.append("=" * 60)
report.append("3. ABLATION TESTS (Table 2)")
report.append("=" * 60)
report.append(f"")
report.append(f"Test                          AUC     Interpretation")
report.append(f"-" * 60)
report.append(f"Verse EARLY vs Verse LATE     {ABLATION_VERSE_VERSE:.2f}    Same genre (chronological signal)")
report.append(f"Verse EARLY vs Prose LATE     {ABLATION_VERSE_PROSE:.2f}    Cross-genre transfer")
report.append(f"Prose vs Prose (SN vs AN)     {ABLATION_PROSE_PROSE:.2f}    Internal prose comparison")

# 4. Test Corpus
report.append("")
report.append("=" * 60)
report.append("4. TEST CORPUS")
report.append("=" * 60)
report.append(f"")
total_segs = len(all_data)
total_suttas = len(sutta_summary)
report.append(f"Total segments: {total_segs:,}")
report.append(f"Total suttas: {total_suttas:,}")
report.append(f"")
report.append(f"Per nikÄya:")
for nik in ['dn', 'mn', 'sn', 'an', 'kn']:
    n_segs = len(all_data[all_data['nikaya'] == nik])
    n_suttas = len(sutta_summary[sutta_summary['nikaya'] == nik])
    report.append(f"  {nik.upper()}: {n_segs:,} segments, {n_suttas} suttas")

# 5. NikÄya Summary (Table 3)
report.append("")
report.append("=" * 60)
report.append("5. NIKÄ€YA SUMMARY (Table 3)")
report.append("=" * 60)
report.append(f"")
report.append(f"NikÄya    Mean P   Median   Std      Suttas    Segments")
report.append(f"-" * 60)
for _, row in NIKAYA_STATS.iterrows():
    report.append(f"{row['nikaya']:<10}{row['mean_p']:.3f}    {row['median_p']:.3f}    {row['std_p']:.3f}    {int(row['n_suttas']):<10}{int(row['n_segments']):,}")

# 6. Non-Circularity Test (Table 4)
report.append("")
report.append("=" * 60)
report.append("6. NON-CIRCULARITY TEST (Table 4)")
report.append("=" * 60)
report.append(f"")
report.append(f"Terms absent from Snp IV-V training data:")
report.append(f"")
report.append(f"Term                          In Snp IV-V   Canon Total   P(early)")
report.append(f"-" * 60)
for _, row in TABLE4.iterrows():
    report.append(f"{row['term']:<30}0             {int(row['n']):<14}{row['mean_p']:.2f}")

# 7. Terminological Substitutions (Table 5)
report.append("")
report.append("=" * 60)
report.append("7. TERMINOLOGICAL SUBSTITUTIONS (Table 5)")
report.append("=" * 60)
report.append(f"")
report.append(f"Archaic Term          N       P       Scholastic Term       N       P       Î”")
report.append(f"-" * 80)
for _, row in TABLE5.iterrows():
    report.append(f"{row['archaic']:<22}{int(row['archaic_n']):<8}{row['archaic_p']:.2f}    {row['scholastic']:<22}{int(row['scholastic_n']):<8}{row['scholastic_p']:.2f}    {row['delta']:.2f}")

# 8. Consciousness Terminology (Table 6)
report.append("")
report.append("=" * 60)
report.append("8. CONSCIOUSNESS TERMINOLOGY (Table 6)")
report.append("=" * 60)
report.append(f"")
report.append(f"Term                              P(early)    N")
report.append(f"-" * 50)
for _, row in TABLE6.iterrows():
    report.append(f"{row['term']:<34}{row['mean_p']:.2f}        {int(row['n'])}")

# 9. Diá¹­á¹­hi Terminology (Table 7)
report.append("")
report.append("=" * 60)
report.append("9. DIá¹¬á¹¬HI TERMINOLOGY (Table 7)")
report.append("=" * 60)
report.append(f"")
report.append(f"Term                              P(early)    N")
report.append(f"-" * 50)
for _, row in TABLE7.iterrows():
    report.append(f"{row['term']:<34}{row['mean_p']:.2f}        {int(row['n'])}")

# 10. Frame Detection
report.append("")
report.append("=" * 60)
report.append("10. FRAME DETECTION")
report.append("=" * 60)
report.append(f"")
report.append(f"'Evaá¹ me sutaá¹' formula: N = {FRAME_EVAM['n']}, Mean P = {FRAME_EVAM['mean_p']:.2f}")
report.append(f"Title/header segments: N = {FRAME_SHORT['n']:,}, Mean P = {FRAME_SHORT['mean_p']:.2f}")

# 11. Khuddaka Breakdown
report.append("")
report.append("=" * 60)
report.append("11. KHUDDAKA NIKÄ€YA BREAKDOWN")
report.append("=" * 60)
report.append(f"")
report.append(f"Collection      Mean P   Median   Segments")
report.append(f"-" * 50)
for coll, row in KN_BREAKDOWN.iterrows():
    report.append(f"{coll:<16}{row['mean']:.3f}    {row['median']:.3f}    {int(row['count']):,}")

# 12. Top Early Suttas
report.append("")
report.append("=" * 60)
report.append("12. TOP 30 EARLY-STYLE SUTTAS (Grey Zone)")
report.append("=" * 60)
report.append(f"")
for _, r in TOP_EARLY_SUTTAS.iterrows():
    report.append(f"{r['sutta_id']:<25} {r['nikaya']:<4} P={r['mean']:.3f} (n={int(r['n_segs'])})")

# 13. Top Late Suttas
report.append("")
report.append("=" * 60)
report.append("13. TOP 30 LATE-STYLE SUTTAS (Grey Zone)")
report.append("=" * 60)
report.append(f"")
for _, r in TOP_LATE_SUTTAS.iterrows():
    report.append(f"{r['sutta_id']:<25} {r['nikaya']:<4} P={r['mean']:.3f} (n={int(r['n_segs'])})")

report.append("")
report.append("=" * 80)
report.append("END OF REPORT")
report.append("=" * 80)

# Save report
report_text = '\n'.join(report)
with open('FINAL_REPORT_COMPLETE.txt', 'w') as f:
    f.write(report_text)

print(report_text)

PÄ€LI CANON STYLOMETRIC ANALYSIS - FINAL REPORT

1. TRAINING CORPORA

EARLY CORE: Aá¹­á¹­hakavagga (Snp 4) + PÄrÄyanavagga (Snp 5) + archaic Snp
  Sources: snp4, snp5, snp1.12, snp1.3, snp3.11, snp3.6
  Segments: 2,406

LATE CORE: Buddhavaá¹ƒsa + Petavatthu + VimÄnavatthu
  Sources: bv, pv, vv
  Segments: 11,212

TOTAL TRAINING: 13,618 segments
Ratio early:late = 1:4.7

EXCLUSIONS: Niddesa texts (mnd, cnd) - canonical commentaries on Snp

2. MODEL PERFORMANCE

MAIN MODEL (5-fold stratified CV):
  AUC = 0.849 Â± 0.006

3. ABLATION TESTS (Table 2)

Test                          AUC     Interpretation
------------------------------------------------------------
Verse EARLY vs Verse LATE     0.85    Same genre (chronological signal)
Verse EARLY vs Prose LATE     0.97    Cross-genre transfer
Prose vs Prose (SN vs AN)     0.93    Internal prose comparison

4. TEST CORPUS

Total segments: 240,353
Total suttas: 5,725

Per nikÄya:
  DN: 16,212 segments, 34 suttas
  MN: 26,775 segments, 152 

## 15. STATISTICAL VALIDATION TESTS (REVIEWER RESPONSE)

In [ ]:
# =============================================================================
# STATISTICAL VALIDATION TESTS
# =============================================================================

# Bootstrap CI function
def bootstrap_ci(values, n_bootstrap=1000, ci=95):
    if len(values) < 3:
        return None, None
    bootstrap_means = [np.mean(np.random.choice(values, size=len(values), replace=True))
                       for _ in range(n_bootstrap)]
    alpha = (100 - ci) / 2
    return np.percentile(bootstrap_means, alpha), np.percentile(bootstrap_means, 100 - alpha)

# Permutation test
def permutation_test(values1, values2, n_perm=10000):
    observed_diff = np.mean(values1) - np.mean(values2)
    combined = np.concatenate([values1, values2])
    n1 = len(values1)
    count = 0
    for _ in range(n_perm):
        np.random.shuffle(combined)
        perm_diff = np.mean(combined[:n1]) - np.mean(combined[n1:])
        if abs(perm_diff) >= abs(observed_diff):
            count += 1
    return count / n_perm

print("✓ Statistical functions ready")

✓ Statistical functions ready


In [ ]:
print("="*70)
print("PERMUTATION TEST: viññāṇaṁ aniccaṁ vs viññāṇaṁ anattā")
print("="*70)

aniccam = all_data[all_data['text'].str.contains('viññāṇaṁ aniccaṁ', case=False, na=False)]['p_early'].values
anatta = all_data[all_data['text'].str.contains('viññāṇaṁ anattā', case=False, na=False)]['p_early'].values

print(f"\nviññāṇaṁ aniccaṁ: N={len(aniccam)}, Mean P={np.mean(aniccam):.3f}")
print(f"viññāṇaṁ anattā: N={len(anatta)}, Mean P={np.mean(anatta):.3f}")

observed_diff = np.mean(aniccam) - np.mean(anatta)
print(f"\nObserved difference: {observed_diff:.3f}")

np.random.seed(42)
p_perm = permutation_test(aniccam, anatta)
print(f"Permutation p-value: {p_perm:.6f}")
print(f"Interpretation: {'SIGNIFICANT' if p_perm < 0.05 else 'NOT significant'} at α=0.05")

PERMUTATION TEST: viññāṇaṁ aniccaṁ vs viññāṇaṁ anattā

viññāṇaṁ aniccaṁ: N=48, Mean P=0.998
viññāṇaṁ anattā: N=38, Mean P=0.617

Observed difference: 0.381
Permutation p-value: 0.000000
Interpretation: SIGNIFICANT at α=0.05


In [ ]:
print("="*70)
print("BOOTSTRAP 95% CI FOR KEY TERMS")
print("="*70)

key_terms = [
    'appatiṭṭhita',
    'tannissitaṁ',
    'viññāṇaṁ aniccaṁ',
    'viññāṇaṁ anattā',
    'sammādiṭṭhi',
    'pabhassara',
]

np.random.seed(42)
print(f"\n{'Term':<25} {'N':<8} {'Mean P':<12} {'95% CI'}")
print("-"*60)

for term in key_terms:
    matches = all_data[all_data['text'].str.contains(term, case=False, na=False)]
    if len(matches) > 0:
        p_vals = matches['p_early'].values
        ci_low, ci_high = bootstrap_ci(p_vals)
        ci_str = f"[{ci_low:.2f}, {ci_high:.2f}]" if ci_low else "N/A"
        print(f"{term:<25} {len(matches):<8} {np.mean(p_vals):<12.3f} {ci_str}")

BOOTSTRAP 95% CI FOR KEY TERMS

Term                      N        Mean P       95% CI
------------------------------------------------------------
appatiṭṭhita              17       0.944        [0.84, 1.00]
tannissitaṁ               17       1.000        [1.00, 1.00]
viññāṇaṁ aniccaṁ          48       0.998        [1.00, 1.00]
viññāṇaṁ anattā           38       0.617        [0.51, 0.72]
sammādiṭṭhi               724      0.701        [0.67, 0.73]
pabhassara                48       0.527        [0.40, 0.66]


In [ ]:
print("="*70)
print("NON-CIRCULARITY VERIFICATION")
print("="*70)

# Get training texts
early_texts = ' '.join([s['text'] for s in early_segs]).lower()
late_texts = ' '.join([s['text'] for s in late_segs]).lower()

check_terms = [
    'appatiṭṭhita',
    'tannissitaṁ',
    'sammādiṭṭhi',
    'viññāṇaṁ aniccaṁ',
    'viññāṇaṁ anattā',
    'bojjhaṅga',
    'rūpaṁ anattā',
]

print(f"\n{'Term':<25} {'In Early':<12} {'In Late':<12} {'Status'}")
print("-"*60)

for term in check_terms:
    in_early = term.lower() in early_texts
    in_late = term.lower() in late_texts
    status = "⚠️ PRESENT" if (in_early or in_late) else "✓ Absent"
    print(f"{term:<25} {str(in_early):<12} {str(in_late):<12} {status}")

NON-CIRCULARITY VERIFICATION

Term                      In Early     In Late      Status
------------------------------------------------------------
appatiṭṭhita              False        False        ✓ Absent
tannissitaṁ               False        False        ✓ Absent
sammādiṭṭhi               False        False        ✓ Absent
viññāṇaṁ aniccaṁ          False        False        ✓ Absent
viññāṇaṁ anattā           False        False        ✓ Absent
bojjhaṅga                 False        False        ✓ Absent
rūpaṁ anattā              False        False        ✓ Absent


In [ ]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, brier_score_loss, precision_recall_curve, auc

print("="*70)
print("CALIBRATION & CLASSIFICATION METRICS")
print("="*70)

# Identify training segments in scored data
EARLY_PATTERNS = ['snp4.', 'snp5.', 'snp1.3', 'snp1.12', 'snp3.6', 'snp3.11']
LATE_PATTERNS = ['bv', 'pv', 'vv']

def is_early(sid):
    sid = str(sid).lower()
    for p in EARLY_PATTERNS:
        if sid.startswith(p): return True
    return False

def is_late(sid):
    sid = str(sid).lower()
    for p in LATE_PATTERNS:
        if sid.startswith(p): return True
    return False

all_data['is_early'] = all_data['sutta_id'].apply(is_early)
all_data['is_late'] = all_data['sutta_id'].apply(is_late)

train_data = all_data[all_data['is_early'] | all_data['is_late']].copy()
y_true = train_data['is_early'].astype(int).values
y_prob = train_data['p_early'].values

# Brier Score
brier = brier_score_loss(y_true, y_prob)
print(f"\nBrier Score: {brier:.4f}")

# AUC-ROC
auc_roc = roc_auc_score(y_true, y_prob)
print(f"AUC-ROC: {auc_roc:.3f}")

# AUC-PR
prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_prob)
auc_pr = auc(rec_curve, prec_curve)
print(f"AUC-PR: {auc_pr:.3f}")

# P>0.95 validation
high_conf = y_prob > 0.95
if high_conf.sum() > 0:
    p95_val = y_true[high_conf].mean()
    print(f"\nP>0.95 validation: {p95_val:.1%} actually early (N={high_conf.sum()})")

# Find optimal threshold
best_f1, best_t = 0, 0.5
for t in np.arange(0.1, 0.9, 0.05):
    y_pred = (y_prob >= t).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
    if f1 > best_f1:
        best_f1, best_t = f1, t

y_pred = (y_prob >= best_t).astype(int)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

print(f"\nAt optimal threshold ({best_t:.2f}):")
print(f"  Precision: {prec:.3f}")
print(f"  Recall: {rec:.3f}")
print(f"  F1: {f1:.3f}")
print(f"\nConfusion Matrix:")
print(f"  TP={tp}, FP={fp}")
print(f"  FN={fn}, TN={tn}")

CALIBRATION & CLASSIFICATION METRICS

Brier Score: 0.0383
AUC-ROC: 0.979
AUC-PR: 0.930

P>0.95 validation: 98.6% actually early (N=1047)

At optimal threshold (0.40):
  Precision: 0.856
  Recall: 0.857
  F1: 0.857

Confusion Matrix:
  TP=2063, FP=348
  FN=343, TN=10864


## 16. Download Files

In [ ]:
from google.colab import files

download_files = [
    'FINAL_REPORT_COMPLETE.txt',
    'all_suttas_summary.csv',
    'dn_scores.csv',
    'mn_scores.csv',
    'sn_scores.csv',
    'an_scores.csv',
    'kn_scores.csv'
]

for f in download_files:
    if os.path.exists(f):
        print(f"Downloading {f}...")
        files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>